# 🧠 NLP for Drilling Engineers
In this session, we'll use NLP techniques to analyze daily operations summaries and automatically detect entries that contain problems (e.g., stuck pipe, losses, etc.).

## 🎯 Objectives
- Understand basic NLP concepts
- Preprocess and clean drilling text data
- Use keyword and ML-based approaches to detect problems
- Optionally classify the type of problem (e.g., stuck pipe, losses)

In [ ]:
# Load necessary libraries
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Load your dataset (adjust path as needed)
df = pd.read_csv('iadc_time_logs_view.csv')
df.head()

In [ ]:
# Load from Google Drive if necessary
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/python-for-drilling-engineers/module_6/iadc_time_logs_view.csv'

df = pd.read_csv(file_path)

In [ ]:
# IADC Code to Name Mapping
iadc_code_to_name = {
    1: 'Rig Up/Down/Move',
    2: 'Drilling',
    3: 'Reaming',
    4: 'Coring',
    5: 'Circulate & Condition Mud',
    6: 'Tripping',
    7: 'Service/Maintain Rig',
    8: 'Repair Rig',
    9: 'Replace Drill Line',
    10: 'Deviation Survey',
    11: 'Wireline Logs',
    12: 'Run Casing & Cement',
    13: 'Wait on Cement',
    14: 'Rig Up/Down BOP',
    15: 'Test BOP',
    16: 'Drill Stem Test',
    17: 'Plug Back',
    18: 'Squeeze Cement',
    19: 'Fishing',
    20: 'Directional Work',
    21: 'Run/Retrieve Riser',
    22: 'Surface Testing',
    23: 'Other',
    24: 'NPT',
    25: 'Operating Status',
    26: 'Safety',
    27: 'Well Control',
    28: 'Coiled Tubing',
    29: 'Perforating',
    30: 'Tubing Trips',
    31: 'Treating & Well Completion',
    32: 'Swabbing',
    33: 'Testing',
    34: 'Subsea Installation'
}

## 🧹 Step 1: Preprocess the Text

In [ ]:
df.info()

In [ ]:
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['clean_text'] = df['details'].astype(str).apply(clean_text)
df[['details', 'clean_text']].head()

### Common Regular Expression Patterns (`re` Library in Python)

| Pattern      | Description                                      |
|--------------|--------------------------------------------------|
| `.`          | Any character except newline                     |
| `\w`         | Word character (letters, digits, underscore)     |
| `\d`         | Digit (0–9)                                      |
| `\s`         | Whitespace (space, tab, newline)                 |
| `\b`         | Word boundary                                    |
| `^`          | Start of string                                  |
| `$`          | End of string                                    |
| `*`          | 0 or more repetitions                            |
| `+`          | 1 or more repetitions                            |
| `?`          | 0 or 1 repetition (optional)                     |
| `{n}`        | Exactly n repetitions                            |
| `{n,}`       | At least n repetitions                           |
| `{n,m}`      | Between n and m repetitions                      |
| `[abc]`      | Any one character of: a, b, or c                 |
| `[^abc]`     | Any character **except**: a, b, or c             |
| `(a|b)`      | Match either a or b                              |
| `(…)`        | Capturing group (used for extraction)            |
| `\\`         | Escape special characters                        |
| `\A`         | Start of string (like `^`, more strict)          |
| `\Z`         | End of string (like `$`, more strict)            |


In [ ]:
# Add IADC code names to the DataFrame
df['iadc_name'] = df['time_code'].map(iadc_code_to_name)

## 🔍 Step 2: Keyword-Based Detection

In [ ]:
keywords = ['stuck', 'lost', 'kick', 'washout', 'plug', 'sidetrack', 'fish']
pattern = '|'.join(keywords)
df['has_problem'] = df['clean_text'].str.contains(pattern, case=False, na=False)
df['keywords'] = df['clean_text'].apply(lambda x: ', '.join([word for word in keywords if word in x]))
df['has_problem'].value_counts()


### Exploring the Results

In [ ]:
lookup_keyword = 'lost'
occurrence_count = df['clean_text'].str.contains(lookup_keyword, case=False, na=False).sum()
print(f"The keyword '{lookup_keyword}' appears {occurrence_count} times in the dataset.")
df[df.keywords.str.contains(lookup_keyword)].head(30)

In [ ]:
def is_valid_lost_circ_entry(text):
    text = text.lower()
    if 'lost' not in text:
        return False
    
    # Include if context matches
    if any(inc in text for inc in include_context):
        # Exclude if unrelated context is present
        if not any(exc in text for exc in exclude_context):
            return True
    return False

# Define context lists
include_context = ['circ', 'circulation', 'returns', 'lcm', 'mud', 'fluid', 'losses', 'formation']
exclude_context = ['generator', 'connection', 'pason', 'communication', 'network', 'internet']

# Apply new logic
df['lost_circ_flag'] = df['clean_text'].apply(is_valid_lost_circ_entry)
df['lost_circ_flag'].value_counts()


In [ ]:
group_df = df[df.has_problem == True].groupby(['time_code', 'iadc_name', 'has_problem']).size()
group_df = group_df.reset_index(name='count')
group_df.sort_values(by='count', ascending=False, inplace=True)
group_df

In [ ]:
group_df = df[df.lost_circ_flag == True].groupby(['time_code', 'iadc_name']).size()
group_df = group_df.reset_index(name='count')
group_df.sort_values(by='count', ascending=False, inplace=True)
group_df

In [ ]:
# Find the most common phrases in the 'details' column for the 'has_problem' == True and 'time_code' == 12
def find_most_common_phrases(df, time_code, n=10):
    filtered_df = df[(df['has_problem'] == True) & (df['time_code'] == time_code)]
    vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')
    X = vectorizer.fit_transform(filtered_df['clean_text'])
    sum_words = X.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

# time_code_list = df[~df.time_code.isna()].time_code.unique().tolist()
time_code_list = [12, 21, 6, 22, 19, 23, 5]
for time_code in time_code_list:
    print(f"Most common phrases for time_code {time_code}-{iadc_code_to_name[time_code]}:")
    common_phrases = find_most_common_phrases(df, time_code)
    print(common_phrases)

## 🔡 Step 3: Convert Text to Features (Bag of Words)

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['clean_text'])
y = df['has_problem']

### Vectorization

- X is the sparse matrix returned by `CountVectorizer`, shape = (documents x vocabulary size)

In [ ]:
X

### Explore single word frequency

In [ ]:
import numpy as np

# Get total frequency of each word
word_counts = np.asarray(X.sum(axis=0)).flatten()
vocab = vectorizer.get_feature_names_out()

# Combine and sort
word_freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
word_freq_df.sort_values(by='count', ascending=False).head(20)


### Using ngram to explore phrases

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Use bigrams and trigrams
ngram_vectorizer = CountVectorizer(ngram_range=(2, 4), max_features=10000)
phrase_df = df[df.has_problem == True].copy()
X_ngrams = ngram_vectorizer.fit_transform(phrase_df['clean_text'])

# Get top phrases
import numpy as np
phrase_counts = np.asarray(X_ngrams.sum(axis=0)).flatten()
phrases = ngram_vectorizer.get_feature_names_out()

phrase_df = pd.DataFrame({'phrase': phrases, 'count': phrase_counts})
phrase_df.sort_values(by='count', ascending=False).head(20)


## 🤖 Step 4: Train a Model to Detect Problems

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

## 🏷️ Optional: Classify Problem Types
If your data has a column like `problem_type`, we can train a multi-class model.

In [ ]:
# Uncomment and adapt if you have a 'problem_type' column
# y_multi = df['problem_type']
# X_train, X_test, y_train, y_test = train_test_split(X, y_multi, test_size=0.2, random_state=42)
# model = LogisticRegression(max_iter=1000, multi_class='multinomial')
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

## ✅ Wrap-Up
- You learned how to clean and process text data
- Used simple rules and ML models to detect problems in ops summaries
- This is just the beginning — this technique can scale across fields, wells, and reporting systems